# Backbones in Practice

**Course:** [Computer Vision](https://ml-viz-ruby.vercel.app/courses/computer-vision/03-backbones-in-practice)

This notebook computes depthwise separable convolution cost savings, visualizes ResNet residual blocks, and compares EfficientNet compound scaling.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Intuition — choosing the workhorse

Every vision system sits on a **backbone** — the feature extractor whose accuracy/latency/size profile
dominates the whole pipeline. Three engineering ideas define the modern menu. **Depthwise separable
convolutions** (MobileNet) factor a standard conv into per-channel spatial + 1×1 mixing, ~8–9× cheaper —
the mobile/edge workhorse. **Residual blocks** (ResNet) add the `+x` identity path, keeping gradient
norms alive at depths where plain stacks die — the default server backbone. **Compound scaling**
(EfficientNet) grows depth, width, and resolution *together* along a measured accuracy-per-FLOP
frontier. This notebook quantifies each with the real numbers you'd use to pick one.

## Depthwise separable convolutions — cost analysis

Standard conv: K² × C_in × C_out multiply-adds per spatial position.  
Depthwise separable: K² × C_in (depthwise) + C_in × C_out (pointwise).

In [ ]:
def conv_cost(K, C_in, C_out):
    """Standard convolution multiply-adds per output pixel."""
    return K * K * C_in * C_out

def depthwise_sep_cost(K, C_in, C_out):
    """Depthwise separable convolution multiply-adds per output pixel."""
    depthwise = K * K * C_in       # one K×K filter per channel
    pointwise = 1 * C_in * C_out   # 1×1 mixing
    return depthwise + pointwise

K = 3
channels = [32, 64, 128, 256, 512]
print(f"{'C_in=C_out':>12} {'Standard':>12} {'DW-Sep':>12} {'Ratio':>8}")
print('-' * 48)
for C in channels:
    std = conv_cost(K, C, C)
    dws = depthwise_sep_cost(K, C, C)
    print(f"{C:>12} {std:>12,} {dws:>12,} {dws/std:>8.3f}")

**What to notice:** the depthwise-separable cost ratio settles near **1/K² ≈ 0.11** as channels grow —
the same `1/C_out + 1/K²` formula verified in the CNN course, now as a deployment table. At `C=512` a
separable layer does ~9× less work than a standard conv of identical receptive field.

In [ ]:
# Visualize reduction ratio as a function of C
C_vals = np.arange(8, 512)
ratios = [depthwise_sep_cost(3, C, C) / conv_cost(3, C, C) for C in C_vals]
theoretical = 1/C_vals + 1/9  # 1/C_out + 1/K^2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(C_vals, ratios, color='#6366f1', linewidth=2, label='Empirical ratio')
ax.plot(C_vals, theoretical, '--', color='#2dd4bf', linewidth=2, label='1/C + 1/9 (theory)')
ax.axhline(1/9, color='#f97316', linestyle=':', alpha=0.7, label='Asymptote 1/9 ≈ 0.111')
ax.set_xlabel('C (in = out channels)')
ax.set_ylabel('Cost ratio (DW-Sep / Standard)')
ax.set_title('Depthwise separable convolution cost reduction (K=3)', fontsize=12)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
ax.set_ylim(0, 0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What to notice:** the savings curve flattens at `1/K²` — the pointwise term's `1/C_out` share
vanishes with width. The efficiency is structural, not a small-model artifact; it holds at any scale,
which is why depthwise separables also appear inside EfficientNet and ConvNeXt.

## ResNet residual blocks — gradient flow analysis

In [ ]:
def simulate_gradient_norm(n_layers, use_residual=True, seed=0):
    """
    Simulate gradient norms as they backpropagate through a deep network.
    Each layer multiplies gradient by a random Jacobian.
    With residuals, gradient = Jacobian_grad + identity_grad.
    """
    np.random.seed(seed)
    grad_norm = 1.0  # start at output
    norms = [grad_norm]
    
    for _ in range(n_layers):
        # Random Jacobian: init to ~ N(0, 1/sqrt(d)) for d=64
        jacobian_scale = np.random.normal(0, 0.9)  # often < 1 → vanishing
        if use_residual:
            grad_norm = abs(jacobian_scale * grad_norm + grad_norm)  # F'(x) + I
        else:
            grad_norm = abs(jacobian_scale * grad_norm)
        norms.append(grad_norm)
    return norms

n_layers = 50
norms_plain = simulate_gradient_norm(n_layers, use_residual=False)
norms_resnet = simulate_gradient_norm(n_layers, use_residual=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, norms, title, color in [
    (axes[0], norms_plain, 'Plain network (no residuals)', '#ef4444'),
    (axes[1], norms_resnet, 'ResNet (with residual shortcuts)', '#2dd4bf')
]:
    ax.semilogy(range(len(norms)), norms, color=color, linewidth=2)
    ax.set_xlabel('Layer (from output backward)')
    ax.set_ylabel('Gradient norm (log scale)')
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What to notice:** on the log axis, the plain network's gradient norm decays exponentially (dead by
layer ~30) while the residual version stays `O(1)` — the `|J + I|` per-layer factor can't fall below
the identity's contribution. Same `+I` Jacobian argument as the modern-architectures lesson, now
simulated over 50 layers.

## The library way — verify the simulation against the exact product

The simulation multiplies random per-layer factors; the claim is statistical. Verify it exactly: the
plain network's gradient norm is `Π|jᵢ|`, and `E[log|j|] < 0` for `j ~ N(0, 0.9)` — so the product
*must* decay exponentially, while the residual factors `|j + 1|` have `E[log|j+1|] > 0`.

In [ ]:
rng = np.random.default_rng(0)
j = rng.normal(0, 0.9, 200_000)
Elog_plain = np.mean(np.log(np.abs(j) + 1e-12))          # plain per-layer factor |j|
Elog_res   = np.mean(np.log(np.abs(j + 1.0) + 1e-12))    # residual per-layer factor |j + 1|
print(f'E[log |j|]     = {Elog_plain:+.3f}   (plain)')
print(f'E[log |j + 1|] = {Elog_res:+.3f}   (residual: the identity shifts the drift UP by {Elog_res-Elog_plain:.2f})')
assert Elog_res - Elog_plain > 0.4, "the identity path must raise the per-layer log-drift substantially"
L = 50
print(f'\nafter {L} layers:  plain ~ 1e{Elog_plain*L/np.log(10):.0f}   vs   residual ~ 1e{Elog_res*L/np.log(10):.0f}')
print('the identity path raises the log-drift far above plain -> gradients survive vastly deeper ✓')

**What to notice:** `E[log|j|] = −0.74` for the plain factor but `E[log|j+1|] = −0.23` for the
residual factor — the `+I` shifts the per-layer log-drift **up by ~0.5**. Over 50 layers that's the
difference between `1e-16` (plain, dead) and `1e-5` (residual, alive) — a *ten-billion-fold* larger
gradient. With realistic near-identity Jacobians the residual drift is ~0; even in this deliberately
harsh toy, the identity path is decisive.

## EfficientNet compound scaling

In [ ]:
# EfficientNet-B0 to B7 specifications
efficientnet_specs = [
    #  phi  resolution  depth_coeff  width_coeff  params(M)  top1
    (0,  224, 1.0,  1.0,   5.3,  77.1),
    (1,  240, 1.1,  1.0,   7.8,  79.1),
    (2,  260, 1.2,  1.1,   9.2,  80.1),
    (3,  300, 1.4,  1.2,  12.0,  81.6),
    (4,  380, 1.8,  1.4,  19.0,  82.9),
    (5,  456, 2.2,  1.6,  30.0,  83.6),
    (6,  528, 2.6,  1.8,  43.0,  84.0),
    (7,  600, 3.1,  2.0,  66.0,  84.3),
]
phi_vals = [s[0] for s in efficientnet_specs]
resolutions = [s[1] for s in efficientnet_specs]
params = [s[4] for s in efficientnet_specs]
top1 = [s[5] for s in efficientnet_specs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Params vs accuracy
axes[0].plot(params, top1, 'o-', color='#6366f1', linewidth=2, markersize=8)
for i, (p, t, phi) in enumerate(zip(params, top1, phi_vals)):
    axes[0].annotate(f'B{phi}', (p, t), textcoords='offset points', xytext=(5, 3), fontsize=9)
axes[0].set_xlabel('Parameters (M)')
axes[0].set_ylabel('ImageNet Top-1 Accuracy (%)')
axes[0].set_title('EfficientNet: accuracy vs model size', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Resolution scaling
axes[1].plot(phi_vals, resolutions, 's-', color='#2dd4bf', linewidth=2, markersize=8, label='Actual')
phi_cont = np.linspace(0, 7, 100)
theory_res = [224 * 1.15**phi for phi in phi_cont]
axes[1].plot(phi_cont, theory_res, '--', color='#f97316', alpha=0.7, label='224 × 1.15^φ (theory)')
axes[1].set_xlabel('φ (compound scaling factor)')
axes[1].set_ylabel('Input resolution')
axes[1].set_title('EfficientNet input resolution scaling', fontsize=11)
axes[1].legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What to notice:** the real EfficientNet B0→B7 table — accuracy climbs from 77.1 to 84.3 as
parameters grow 5.3M→66M, with the *balanced* growth of depth/width/resolution keeping every model on
the accuracy-per-FLOP frontier. Note the diminishing returns after B4: +36M parameters buys the last
+1.4 points.

## Gotchas & tradeoffs

- **FLOPs ≠ latency.** Depthwise convs have low arithmetic intensity and can be *memory-bound* on GPU —
  a "9× cheaper" layer is often only 2–3× faster. Benchmark on the target hardware.
- **The backbone table is the menu, not the answer:** mobile → MobileNet/EfficientNet-lite; server
  accuracy → ResNet/ConvNeXt/ViT; the right pick depends on the deployment budget.
- **Higher input resolution is quadratic cost** — EfficientNet's resolution axis is the most expensive
  one to scale.
- **Pretrained weights dominate:** whichever backbone you pick, ImageNet (or CLIP) initialization beats
  architecture tweaks at typical data scales (the transfer-learning lesson).

In [ ]:
# FLOPs vs practical speed: arithmetic intensity (ops per byte moved) is what GPUs care about
K, C = 3, 256
std_ops = K*K*C*C
dws_ops = K*K*C + C*C
# bytes moved (weights + activations, float32, per output pixel — rough model)
std_bytes = 4 * (K*K*C*C / (56*56) + 2*C)      # weights amortized over a 56x56 map + in/out activations
dws_bytes = 4 * ((K*K*C + C*C) / (56*56) + 3*C)
print(f'standard conv : {std_ops:>8,} ops, intensity ~ {std_ops/std_bytes:6.0f} ops/byte')
print(f'depthwise sep : {dws_ops:>8,} ops, intensity ~ {dws_ops/dws_bytes:6.0f} ops/byte')
print('\n-> the separable conv does ~9x less math but also has much lower ops/byte,')
print('   so on bandwidth-limited hardware the wall-clock speedup is far below 9x')

**What to notice:** the separable layer's arithmetic intensity is a fraction of the standard conv's —
it moves nearly as many bytes while doing far less math, so memory bandwidth, not FLOPs, sets its
speed on GPUs. This FLOPs-vs-latency gap (formalized by the roofline model in the GPU course) is why
backbone choices must be benchmarked, not paper-mathed.

## ✏️ Your turn

### Exercise 1: Compute depthwise separable conv savings

Implement a function that returns the FLOPs ratio (depthwise separable / standard) for given convolution parameters.

In [ ]:
def dw_sep_ratio(K, C_in, C_out):
    """
    Compute the FLOPs ratio: depthwise_sep_cost / standard_conv_cost.
    
    Args:
        K: int, kernel size (K×K)
        C_in: int, input channels
        C_out: int, output channels
    Returns:
        float: ratio in (0, 1), lower = more efficient
    """
    # TODO(you): compute both costs and return the ratio
    # Standard: K^2 * C_in * C_out
    # DW-Sep: K^2 * C_in + C_in * C_out
    pass


print(f"K=3, C_in=256, C_out=256: {dw_sep_ratio(3, 256, 256):.4f}")
print(f"K=3, C_in=32, C_out=32:   {dw_sep_ratio(3, 32, 32):.4f}")

In [ ]:
r = dw_sep_ratio(3, 256, 256)
assert r is not None, "Should return a value"
assert 0.10 < r < 0.15, f"Expected ~0.115 for K=3,C=256, got {r}"
# Ratio should be close to 1/C + 1/K^2 = 1/256 + 1/9 ≈ 0.115
theoretical = 1/256 + 1/9
assert abs(r - theoretical) < 0.002, f"Should match theory 1/C+1/K², got {r} vs {theoretical:.4f}"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def dw_sep_ratio(K, C_in, C_out):
    standard = K * K * C_in * C_out
    dw_sep = K * K * C_in + C_in * C_out
    return dw_sep / standard
```
</details>

### Exercise 2: Implement a residual block forward pass

Given input x and a 'layer' function F, implement the residual block output F(x) + x.

In [ ]:
def residual_block(x, F):
    """
    Compute the output of a residual block: F(x) + x.
    
    Args:
        x: np.ndarray, input tensor
        F: callable, the residual function (two conv layers in practice)
    Returns:
        np.ndarray: output of same shape as x
    """
    # TODO(you): apply F(x) and add the skip connection
    pass


# Test with identity function F(x) = 0 (zero initialization)
x_test = np.array([1.0, 2.0, 3.0, 4.0])
F_zero = lambda x: np.zeros_like(x)
F_scale = lambda x: 0.5 * x

print("F(x) = 0 (zero residual):")
print(f"  Output: {residual_block(x_test, F_zero)}  (should equal x)")
print("\nF(x) = 0.5x:")
print(f"  Output: {residual_block(x_test, F_scale)}  (should be 1.5x)")

In [ ]:
x_test = np.array([1.0, 2.0, 3.0, 4.0])
out_zero = residual_block(x_test, lambda x: np.zeros_like(x))
assert out_zero is not None, "Should return a value"
np.testing.assert_allclose(out_zero, x_test, err_msg="F(x)=0 → output should equal x (identity)")
out_scale = residual_block(x_test, lambda x: 0.5 * x)
np.testing.assert_allclose(out_scale, 1.5 * x_test, err_msg="F(x)=0.5x → output should be 1.5x")
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def residual_block(x, F):
    return F(x) + x
```

The elegance of residual learning: if F learns to output near-zero, the block approximates the identity — it's easy for the network to learn "keep this feature as-is" by driving F toward zero.
</details>